# Pose Normalization

Per-frame normalization using shoulder center as origin and torso size as scale.

**Input:** `keypoints_interpolated_15_boundary`  
**Output:** shoulder center = (0, 0), torso size = 1

In [70]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

IN_DIR  = Path('../data/processed/keypoints_interpolated_15_boundary')
OUT_DIR = Path('../data/processed/keypoints_normalized')

MODELS = [
    ('movenet',        'confidence'),
    ('mediapipe_norm', 'visibility'),
]

JOINTS = [
    'left_shoulder', 'right_shoulder',
    'left_elbow',    'right_elbow',
    'left_wrist',    'right_wrist',
    'left_hip',      'right_hip',
    'left_knee',     'right_knee',
    'left_ankle',    'right_ankle',
]

## Normalization function

Frames with missing reference joints (shoulder or hip) are left as NaN.

In [71]:
def normalize_pose(df):
    df = df.copy()

    sc_x = (df['left_shoulder_x'] + df['right_shoulder_x']) / 2
    sc_y = (df['left_shoulder_y'] + df['right_shoulder_y']) / 2

    hc_x = (df['left_hip_x'] + df['right_hip_x']) / 2
    hc_y = (df['left_hip_y'] + df['right_hip_y']) / 2

    torso_size = np.sqrt((sc_x - hc_x) ** 2 + (sc_y - hc_y) ** 2)
    invalid = (torso_size == 0) | torso_size.isna()

    for joint in JOINTS:
        df[f'{joint}_x'] = (df[f'{joint}_x'] - sc_x) / torso_size
        df[f'{joint}_y'] = (df[f'{joint}_y'] - sc_y) / torso_size
        df.loc[invalid, f'{joint}_x'] = np.nan
        df.loc[invalid, f'{joint}_y'] = np.nan

    return df

In [72]:
for model, _ in MODELS:
    out_model = OUT_DIR / model
    out_model.mkdir(parents=True, exist_ok=True)
    print(f'=== {model} ===')

    for csv_path in sorted((IN_DIR / model).glob('*.csv')):
        df      = pd.read_csv(csv_path)
        df_norm = normalize_pose(df)
        df_norm.to_csv(out_model / csv_path.name, index=False)
        print(f'  {csv_path.stem}: {len(df)} frames')
    print()

=== movenet ===
  DJI_20250425092743_0028_D_movenet: 94 frames
  DJI_20250425093100_0030_D_movenet: 92 frames


  DJI_20250425104507_0045_D_movenet: 73 frames
  DJI_20250425104804_0047_D_movenet: 94 frames
  DJI_20250425112502_0059_D_movenet: 123 frames
  DJI_20250425112749_0061_D_movenet: 151 frames
  DJI_20250425120835_0074_D_movenet: 60 frames
  DJI_20250425121226_0076_D_movenet: 151 frames
  DJI_20250425125202_0091_D_movenet: 61 frames
  DJI_20250425125448_0093_D_movenet: 91 frames

=== mediapipe_norm ===
  DJI_20250425092743_0028_D_mediapipe_norm: 94 frames
  DJI_20250425093100_0030_D_mediapipe_norm: 92 frames
  DJI_20250425104507_0045_D_mediapipe_norm: 73 frames
  DJI_20250425104804_0047_D_mediapipe_norm: 94 frames
  DJI_20250425112502_0059_D_mediapipe_norm: 123 frames
  DJI_20250425112749_0061_D_mediapipe_norm: 151 frames
  DJI_20250425120835_0074_D_mediapipe_norm: 60 frames
  DJI_20250425121226_0076_D_mediapipe_norm: 151 frames
  DJI_20250425125202_0091_D_mediapipe_norm: 61 frames
  DJI_20250425125448_0093_D_mediapipe_norm: 91 frames



## Verify

Shoulder center should be (0, 0) in every normalized frame.

In [73]:
for model, _ in MODELS:
    csv_path = sorted((OUT_DIR / model).glob('*.csv'))[0]
    df = pd.read_csv(csv_path).dropna(subset=['left_shoulder_x'])

    sc_x = (df['left_shoulder_x'] + df['right_shoulder_x']) / 2
    sc_y = (df['left_shoulder_y'] + df['right_shoulder_y']) / 2

    print(f'{model}: {csv_path.stem}')
    print(f'  shoulder center x — mean: {sc_x.mean():.6f}, max abs: {sc_x.abs().max():.6f}')
    print(f'  shoulder center y — mean: {sc_y.mean():.6f}, max abs: {sc_y.abs().max():.6f}')
    print()

movenet: DJI_20250425092743_0028_D_movenet
  shoulder center x — mean: 0.000000, max abs: 0.000000
  shoulder center y — mean: 0.000000, max abs: 0.000000

mediapipe_norm: DJI_20250425092743_0028_D_mediapipe_norm
  shoulder center x — mean: -0.000000, max abs: 0.000000
  shoulder center y — mean: -0.000000, max abs: 0.000000



## Step 6: Temporal Stability

Frame-to-frame Euclidean displacement per joint in normalized space.


Lower = smoother tracking.

### Stability per video

In [74]:
rows = []

for model, _ in MODELS:
    for csv_path in sorted((OUT_DIR / model).glob('*.csv')):
        df = pd.read_csv(csv_path)
        disps = []
        for joint in JOINTS:
            x = df[f'{joint}_x'].values
            y = df[f'{joint}_y'].values
            d = np.sqrt(np.diff(x) ** 2 + np.diff(y) ** 2)
            disps.extend(d[~np.isnan(d)])
        rows.append({
            'Video': csv_path.stem,
            'Model': model,
            'Mean Displacement': round(np.mean(disps), 4),
            'Std Displacement':  round(np.std(disps), 4),
        })

df_overall = pd.DataFrame(rows)
df_overall.to_csv(OUT_DIR / 'stability_per_video.csv', index=False)
df_overall

,Video,Model,Mean Displacement,Std Displacement
0,DJI_20250425092743_0028_D_movenet,movenet,0.0192,0.0151
1,DJI_20250425093100_0030_D_movenet,movenet,0.0221,0.0209
2,DJI_20250425104507_0045_D_movenet,movenet,0.0199,0.0157
3,DJI_20250425104804_0047_D_movenet,movenet,0.0180,0.0181
4,DJI_20250425112502_0059_D_movenet,movenet,0.0155,0.0120
5,DJI_20250425112749_0061_D_movenet,movenet,0.0147,0.0138
6,DJI_20250425120835_0074_D_movenet,movenet,0.0181,0.0146
7,DJI_20250425121226_0076_D_movenet,movenet,0.0147,0.0138
8,DJI_20250425125202_0091_D_movenet,movenet,0.0221,0.0185
9,DJI_20250425125448_0093_D_movenet,movenet,0.0151,0.0180


### Per-joint stability

To identifying which joints have higher jitter.

In [75]:
rows_joint = []

for model, _ in MODELS:
    for csv_path in sorted((OUT_DIR / model).glob('*.csv')):
        df = pd.read_csv(csv_path)
        for joint in JOINTS:
            x = df[f'{joint}_x'].values
            y = df[f'{joint}_y'].values
            d = np.sqrt(np.diff(x) ** 2 + np.diff(y) ** 2)
            d = d[~np.isnan(d)]
            if len(d) == 0:
                continue
            rows_joint.append({
                'Video': csv_path.stem,
                'Model': model,
                'Joint': joint,
                'Mean Displacement': round(np.mean(d), 4),
                'Std Displacement':  round(np.std(d), 4),
            })

df_joint = pd.DataFrame(rows_joint)
df_joint.to_csv(OUT_DIR / 'stability_per_joint.csv', index=False)
df_joint

,Video,Model,Joint,Mean Displacement,Std Displacement
0,DJI_20250425092743_0028_D_movenet,movenet,left_shoulder,0.0090,0.0046
1,DJI_20250425092743_0028_D_movenet,movenet,right_shoulder,0.0090,0.0046
2,DJI_20250425092743_0028_D_movenet,movenet,left_elbow,0.0146,0.0070
3,DJI_20250425092743_0028_D_movenet,movenet,right_elbow,0.0193,0.0099
4,DJI_20250425092743_0028_D_movenet,movenet,right_wrist,0.0133,0.0063
...,...,...,...,...,...
227,DJI_20250425125448_0093_D_mediapipe_norm,mediapipe_norm,right_hip,0.0053,0.0046
228,DJI_20250425125448_0093_D_mediapipe_norm,mediapipe_norm,left_knee,0.0104,0.0115
229,DJI_20250425125448_0093_D_mediapipe_norm,mediapipe_norm,right_knee,0.0111,0.0124
230,DJI_20250425125448_0093_D_mediapipe_norm,mediapipe_norm,left_ankle,0.0136,0.0168


## Step 7: Standardization

In [76]:
from sklearn.preprocessing import StandardScaler

STD_DIR = Path('../data/processed/keypoints_standardized')
STD_DIR.mkdir(parents=True, exist_ok=True)

COORD_COLS = []

for joint in JOINTS:
    COORD_COLS.append(joint + "_x")
    COORD_COLS.append(joint + "_y")

for model, _ in MODELS:

    print(f"\n=== {model} ===")

    model_input_path = OUT_DIR / model
    
    model_output_path = STD_DIR / model
    model_output_path.mkdir(parents=True, exist_ok=True)

    all_frames_list = []

    for csv_file in model_input_path.glob("*.csv"):
        df = pd.read_csv(csv_file)
        all_frames_list.append(df[COORD_COLS])

    full_dataset = pd.concat(all_frames_list, ignore_index=True)

    scaler = StandardScaler()
    scaler.fit(full_dataset.dropna())


    for csv_file in sorted(model_input_path.glob("*.csv")):

        df = pd.read_csv(csv_file)
        valid_rows = df[COORD_COLS].notna().all(axis=1)

        if valid_rows.any():
            df.loc[valid_rows, COORD_COLS] = scaler.transform(
                df.loc[valid_rows, COORD_COLS]
            )

        df.to_csv(model_output_path / csv_file.name, index=False)
        print(f"{csv_file.name} -> standardized")


=== movenet ===
DJI_20250425092743_0028_D_movenet.csv -> standardized


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Library/Frameworks/Python.framework/Versions/3.9

DJI_20250425093100_0030_D_movenet.csv -> standardized
DJI_20250425104507_0045_D_movenet.csv -> standardized
DJI_20250425104804_0047_D_movenet.csv -> standardized
DJI_20250425112502_0059_D_movenet.csv -> standardized
DJI_20250425112749_0061_D_movenet.csv -> standardized
DJI_20250425120835_0074_D_movenet.csv -> standardized
DJI_20250425121226_0076_D_movenet.csv -> standardized


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():


DJI_20250425125202_0091_D_movenet.csv -> standardized
DJI_20250425125448_0093_D_movenet.csv -> standardized

=== mediapipe_norm ===


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Library/Frameworks/Python.framework/Versions/3.9

DJI_20250425092743_0028_D_mediapipe_norm.csv -> standardized
DJI_20250425093100_0030_D_mediapipe_norm.csv -> standardized
DJI_20250425104507_0045_D_mediapipe_norm.csv -> standardized
DJI_20250425104804_0047_D_mediapipe_norm.csv -> standardized
DJI_20250425112502_0059_D_mediapipe_norm.csv -> standardized
DJI_20250425112749_0061_D_mediapipe_norm.csv -> standardized
DJI_20250425120835_0074_D_mediapipe_norm.csv -> standardized
DJI_20250425121226_0076_D_mediapipe_norm.csv -> standardized
DJI_20250425125202_0091_D_mediapipe_norm.csv -> standardized
DJI_20250425125448_0093_D_mediapipe_norm.csv -> standardized


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/utils/validation.py:623: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
